# 03 - Knowledge Distillation ResNet (CIFAR-10, <5M params) - FINAL MODEL

**Best submission: public leaderboard 0.901.** This is the final model.

**Teacher (training aid, not submitted):** ResNet blocks [4,4,4], channels [96,192,384], 13,193,610 params. Trained 160 epochs with AutoAugment + label smoothing. Reached 96.90% validation.

**Student (submitted, <5M):** ResNet blocks [4,4,3], channels [64,128,256], 4,697,162 params. Trained 250 epochs with temperature knowledge distillation (T=4, alpha=0.7) from the teacher's soft predictions, plus CutMix/Mixup, 5-epoch warmup then cosine, gradient clipping. Reached 97.00% validation, public leaderboard **0.901**.

**Inference:** flip-only test-time augmentation. (10-crop TTA was tested and *reduced* the public score on this eval set, from 0.901 to 0.888, because reflect-pad crops add artifacts that do not match the shifted eval images.)

**Why distillation:** only the single student is submitted, so the model stays under the 5M cap while learning from a larger teacher. This gave the best accuracy of all approaches tried (0.865 baseline -> 0.883 AutoAugment -> 0.901 distillation).

Compliance: trained from scratch on the permitted data only, no pretrained weights, no external data.

Run order on Kaggle: GPU on, attach the competition data, run all. ~4 to 5 hours total (commit to run headless).

In [ ]:
import functools; print=functools.partial(print, flush=True)
import os, glob, time, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
DEVICE="cuda" if torch.cuda.is_available() else "cpu"; assert DEVICE=="cuda","Turn on GPU."
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

In [ ]:
def _f(f): h=glob.glob(f"/kaggle/input/**/{f}",recursive=True); return h[0] if h else None
def _d(h):
    for dp,dn,fn in os.walk("/kaggle/input"):
        if h in os.path.basename(dp).lower() and any(x.lower().endswith(".png") for x in fn[:50]): return dp
    return None
TRAIN_CSV,TRAIN_DIR=_f("train_labels.csv"),_d("train")
TEST_CSV,TEST_DIR,SAMPLE=_f("test.csv"),_d("test"),_f("sample_submission.csv")
assert all([TRAIN_CSV,TRAIN_DIR,TEST_CSV,TEST_DIR,SAMPLE]),"Data not attached (+ Add Input the competition)."

In [ ]:
MEAN,STD=(0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616)
train_tf=T.Compose([T.RandomCrop(32,padding=4),T.RandomHorizontalFlip(),T.AutoAugment(T.AutoAugmentPolicy.CIFAR10),T.ToTensor(),T.Normalize(MEAN,STD),T.RandomErasing(p=0.5)])
eval_tf=T.Compose([T.ToTensor(),T.Normalize(MEAN,STD)])
def preload(df,d):
    a=np.zeros((len(df),32,32,3),np.uint8)
    for i,fn in enumerate(df["filename"].values): a[i]=np.array(Image.open(os.path.join(d,fn)).convert("RGB"))
    return a
class DS(Dataset):
    def __init__(s,im,me,tf): s.im,s.me,s.tf=im,np.asarray(me),tf
    def __len__(s): return len(s.im)
    def __getitem__(s,i): return s.tf(Image.fromarray(s.im[i])), int(s.me[i])
df=pd.read_csv(TRAIN_CSV); print("preloading train ..."); imgs=preload(df,TRAIN_DIR); lab=df["label"].values
perm=np.random.default_rng(SEED).permutation(len(df)); nv=int(len(df)*0.02); vi,ti=perm[:nv],perm[nv:]
train_loader=DataLoader(DS(imgs[ti],lab[ti],train_tf),batch_size=128,shuffle=True,num_workers=4,pin_memory=True,drop_last=True,persistent_workers=True)
val_loader=DataLoader(DS(imgs[vi],lab[vi],eval_tf),batch_size=512,shuffle=False,num_workers=4,pin_memory=True,persistent_workers=True)
print(f"train={len(ti)}  val={len(vi)}")

In [ ]:
class BasicBlock(nn.Module):
    def __init__(s,ic,oc,k=3,st=1):
        super().__init__(); p=k//2
        s.c1=nn.Conv2d(ic,oc,k,st,p,bias=False); s.b1=nn.BatchNorm2d(oc)
        s.c2=nn.Conv2d(oc,oc,k,1,p,bias=False); s.b2=nn.BatchNorm2d(oc)
        s.sc=nn.Sequential() if (st==1 and ic==oc) else nn.Sequential(nn.Conv2d(ic,oc,1,st,bias=False),nn.BatchNorm2d(oc))
    def forward(s,x): o=F.relu(s.b1(s.c1(x))); o=s.b2(s.c2(o)); return F.relu(o+s.sc(x))
class ResNet(nn.Module):
    def __init__(s,bl,ch,stem=64,k=3,nc=10):
        super().__init__(); s.stem=nn.Sequential(nn.Conv2d(3,stem,3,1,1,bias=False),nn.BatchNorm2d(stem),nn.ReLU(True))
        L,ic=[],stem
        for i,(nb,oc) in enumerate(zip(bl,ch)):
            st=1 if i==0 else 2
            for b in range(nb): L.append(BasicBlock(ic,oc,k,st if b==0 else 1)); ic=oc
        s.layers=nn.Sequential(*L); s.head=nn.Linear(ic,nc)
    def forward(s,x): x=s.stem(x); x=s.layers(x); return s.head(F.adaptive_avg_pool2d(x,1).flatten(1))
cp=lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
@torch.no_grad()
def vacc(m):
    m.eval(); c=t=0
    for x,y in val_loader:
        x,y=x.to(DEVICE),y.to(DEVICE)
        with torch.amp.autocast("cuda"): o=m(x)
        c+=(o.argmax(1)==y).sum().item(); t+=y.size(0)
    return c/max(t,1)

In [ ]:
def rbb(W,H,lam):
    r=np.sqrt(1-lam); cw,ch=int(W*r),int(H*r); cx,cy=np.random.randint(W),np.random.randint(H)
    return np.clip(cx-cw//2,0,W),np.clip(cy-ch//2,0,H),np.clip(cx+cw//2,0,W),np.clip(cy+ch//2,0,H)
T_KD,ALPHA,WARM,CLIP=4.0,0.7,5,5.0
def train(m,epochs,teacher=None,mixup=False,ls=0.1,tag=""):
    ce=nn.CrossEntropyLoss(label_smoothing=ls)
    opt=torch.optim.SGD(m.parameters(),lr=0.1,momentum=0.9,weight_decay=5e-4,nesterov=True)
    sch=torch.optim.lr_scheduler.SequentialLR(opt,
        [torch.optim.lr_scheduler.LinearLR(opt,0.1,1.0,WARM),
         torch.optim.lr_scheduler.CosineAnnealingLR(opt,epochs-WARM)],[WARM])
    scaler=torch.amp.GradScaler("cuda"); best,bs=0.0,None
    for ep in range(epochs):
        m.train(); t0=time.time()
        for x,y in train_loader:
            x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad(set_to_none=True)
            ya=yb=y; lam=1.0
            if mixup:
                lam=np.random.beta(1.0,1.0); idx=torch.randperm(x.size(0),device=x.device); ya,yb=y,y[idx]
                if random.random()<0.5:
                    x=x.clone(); a,b,c,d=rbb(32,32,lam); x[:,:,b:d,a:c]=x[idx,:,b:d,a:c]; lam=1-((c-a)*(d-b)/1024)
                else: x=lam*x+(1-lam)*x[idx]
            with torch.amp.autocast("cuda"):
                out=m(x)
                if teacher is not None:
                    with torch.no_grad(): tout=teacher(x)
            hard=lam*ce(out,ya)+(1-lam)*ce(out,yb)
            if teacher is not None:
                kd=F.kl_div(F.log_softmax(out.float()/T_KD,1),F.softmax(tout.float()/T_KD,1),reduction="batchmean")*(T_KD*T_KD)
                loss=ALPHA*kd+(1-ALPHA)*hard
            else: loss=hard
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(m.parameters(),CLIP); scaler.step(opt); scaler.update()
        sch.step(); acc=vacc(m)
        if acc>best: best=acc; bs={k:v.detach().cpu().clone() for k,v in m.state_dict().items()}
        if ep%10==0 or ep==epochs-1: print(f"[{tag}] ep {ep:3d}  val {acc*100:5.2f}%  best {best*100:5.2f}%  {time.time()-t0:4.1f}s")
    m.load_state_dict(bs); return best

In [ ]:
# Teacher: larger ResNet (13.2M params, never submitted)
teacher=ResNet([4,4,4],[96,192,384]).to(DEVICE); print("teacher params:",f"{cp(teacher):,}")
tb=train(teacher,160,teacher=None,mixup=False,ls=0.1,tag="teacher")
torch.save({"state_dict":teacher.state_dict()},"teacher.pt"); teacher.eval()
for p in teacher.parameters(): p.requires_grad=False
print("teacher best",f"{tb*100:.2f}%")

In [ ]:
# Student: submitted model, blocks [4,4,3] (4.70M params), distilled + CutMix/Mixup
student=ResNet([4,4,3],[64,128,256]).to(DEVICE); ns=cp(student); print("student params:",f"{ns:,}"); assert ns<=5_000_000
sb=train(student,250,teacher=teacher,mixup=True,ls=0.0,tag="student")
torch.save({"state_dict":student.state_dict(),"val_acc":sb,"n_params":ns},"distilled_student.pt")
print("student best",f"{sb*100:.2f}%")

In [ ]:
# Flip-only test-time augmentation (10-crop was tested and hurt on this eval set)
tdf=pd.read_csv(TEST_CSV); sdf=pd.read_csv(SAMPLE); print("preloading test ..."); tnp=preload(tdf,TEST_DIR)
tl=DataLoader(DS(tnp,tdf["id"].values,eval_tf),batch_size=512,shuffle=False,num_workers=4,pin_memory=True)
student.eval(); ids=[]; pr=[]
with torch.no_grad():
    for x,ix in tl:
        x=x.to(DEVICE)
        with torch.amp.autocast("cuda"): lg=student(x)+student(torch.flip(x,dims=[3]))
        pr.append(lg.argmax(1).cpu().numpy()); ids.append(ix.numpy())
ids=np.concatenate(ids); pr=np.concatenate(pr); pm=dict(zip(ids.tolist(),pr.tolist()))
ic,lc=sdf.columns[0],sdf.columns[1]; out=sdf.copy(); out[lc]=out[ic].map(pm).astype(int)
assert out[lc].notna().all(); out.to_csv("submission.csv",index=False); print("wrote submission.csv"); print(out[lc].value_counts().sort_index())